# Partitioning and Caching

**Partitioning** is crucial because it allows Spark to distribute the data envenly across available executors. <br>
This setups can signficantly reduce processing time by enabling parallel operations

**Caching** stores the data in memory, so when you perform multiple actions on the same dataset - such as filtering and aggregating - the data is readily available, reducing the need for re-computation


# Summary: Partitioning & Caching — Classic vs Serverless

## Lazy evaluation (the foundation)

Spark does **not** store your data in memory just because you defined a DataFrame.
A DataFrame is only a *logical plan* (a recipe). No CSV is read until an **action** (`count()`, `show()`, `write()`) runs — and when it does, the whole lineage (read → group → transform) executes in a **single pass**. The cost of "no caching" is not transformations, it is **repeated actions**: every new action re-executes everything from the source file.

## Partitioning

**What it does:** splits data into parallel chunks across executors — it controls *parallelism*, not storage. Data still lives in cloud storage (S3).

* Without it: Spark picks default partitioning (e.g. number of file splits / `spark.sql.shuffle.partitions`), which can be uneven.
* With `repartition(10)`: the 2M rows were split almost perfectly evenly (~199,999–200,001 rows per partition), enabling predictable parallel work per executor.

## Caching / Materialization

| Aspect | Classic compute (`.cache()`) | Serverless (this notebook) |
| --- | --- | --- |
| Store materialized results in memory | `df.cache()` / `df.persist()` | Not supported — raises `PERSIST TABLE is not supported on serverless` |
| Check partition count | `df.rdd.getNumPartitions()` | `df.select(F.spark_partition_id()).distinct().count()` (RDD layer unavailable via Spark Connect) |
| Repeated actions on same DataFrame | Hit the in-memory cache | Re-execute lineage — OR materialize to a **temporary table**: `CREATE OR REPLACE TEMPORARY TABLE ... AS SELECT ...` |
| Transparent caching | Manual (and disk cache) | **Automatic disk (IO) cache**: files read from remote storage are cached on local SSD, no code needed |

## Key takeaways

* Transformations stack in the plan; only actions cost a scan — and each action re-reads the source unless results are materialized.
* `repartition(n)` buys even parallelism; it does not persist anything.
* On serverless, replace `cache()` + `count()` with a temp table (or rely on the automatic disk cache for raw file re-reads).
* Clean up temp tables when done — unlike memory cache, they are not evicted automatically.

In [0]:
# Loading the 2M rows file
file_path = '/Volumes/workspace/bronze_wwi/raw_data/2000000 BT Records.csv'
df_cc = spark.read.format('csv').option('header', 'true').load(file_path)

# Print colummn name and row count
total_rows = df_cc.count()
print('Column names:', df_cc.columns)
print('Total records in the dataset: ', total_rows)

In [0]:
from pyspark.sql import functions as F

# Repartitioning the dataset
df_cc_repart = df_cc.repartition(10)

# On serverless (Spark Connect) the .rdd accessor is not available, df.rdd.getNumPartitions()
# so count distinct partition IDs using spark_partition_id() instead
num_partitions = df_cc_repart.select(
    F.spark_partition_id().alias('partition_id')
).distinct().count()

print('Number of partitions: ', num_partitions)

# Bonus: rows per partition
print('\nRows per partition:')
df_cc_repart.groupBy(F.spark_partition_id().alias('partition_id')).count().orderBy('partition_id').show()
#

In [0]:
# Caching is not available at Databricks Free edition
# Cache the repartitioned DataFrame to boost performance for iterative operations
## df_cc_repart.cache()

# Trigger caching by counting the records
## cached_count = df_cc_repart.count()
## print('Total records after caching (should match): ', cached_count)

In [0]:
# Any transformations applied in the cached_dataframe will run faster then directly transforming the data from the file

In [0]:
# Unpersist the cached DataFrame to free up resouces after transformations
## df_cc_repart.unpersist

# Bonus: Does it make sense to cache a DataFrame that reads from a Delta table?

**Usually no** — and on serverless compute it is not even an option. Here is why:

## Delta reads are already cheap by design

* **Columnar Parquet**: Spark reads only the columns the query touches, unlike the CSV in this notebook where every scan re-parses all fields.
* **Data skipping**: the Delta transaction log lets filtered queries skip whole files (via stats, liquid clustering, or partition pruning) — often touching a small fraction of the table.
* **Automatic disk (IO) cache**: on serverless and classic compute, repeated Delta file reads hit local SSD automatically, so the raw I/O benefit of `cache()` is largely already provided.

## When caching/materialization *does* make sense

* **Cache derived results, not raw reads** — if an expensive aggregation, join, or `repartition` output is reused repeatedly, materializing that result skips recomputing the whole lineage. That is the real win.
* **Iterative algorithms** — ML training loops that iterate over the same rows many times.
* **Slow sources** — JDBC, APIs, or unoptimized CSV/JSON where re-parsing dominates runtime.

## Rule of thumb

> Do not cache a table read; cache (or materialize, on serverless) the **expensive computation built on top of it** — and only if you will reuse it.

In this notebook, the equivalent is materializing `df_cc_repart` into a temp table: what repeats is the CSV read + repartition shuffle, not a Delta scan.